<a href="https://colab.research.google.com/github/Rodriamarog/InteligenciaComputacional/blob/main/AsistenteDigitalPersonal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalar Paqueterias

In [1]:
!pip install faiss-cpu langchain langchain-core langchain-text-splitters \
          sentence-transformers openai numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.4 MB/s eta 0:00:00


Imports y configs

In [2]:
import os, glob as globmod
from typing import Any
import numpy as np
import faiss
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from openai import OpenAI

DEFAULT_DATA_DIR       = "data"
DEFAULT_EMBEDDING_MODEL = "all-MiniLM-L6-v2"
DEFAULT_LLM_MODEL       = "gpt-4.1-mini"
DEFAULT_CHUNK_SIZE      = 256
DEFAULT_CHUNK_OVERLAP   = 32
DEFAULT_TOP_K           = 4
DOC_TYPE_MAP = {"emails":"email","notes":"note","sms":"sms","calendar":"calendar"}

Configuracion

In [3]:
def _parse_int_setting(name, value):
    try: return int(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{name} must be an integer; got {value!r}") from exc

def resolve_config(config=None):
    config = config or {}
    resolved = {
        "api_key":         config.get("api_key",         os.environ.get("OPENAI_API_KEY")),
        "base_url":        config.get("base_url",        os.environ.get("OPENAI_BASE_URL")),
        "model":           config.get("model",           os.environ.get("LLM_MODEL", DEFAULT_LLM_MODEL)),
        "embedding_model": config.get("embedding_model", os.environ.get("EMBEDDING_MODEL", DEFAULT_EMBEDDING_MODEL)),
        "top_k":      _parse_int_setting("TOP_K",      config.get("top_k",      os.environ.get("TOP_K",      DEFAULT_TOP_K))),
        "chunk_size":  _parse_int_setting("CHUNK_SIZE",  config.get("chunk_size",  os.environ.get("CHUNK_SIZE",  DEFAULT_CHUNK_SIZE))),
        "chunk_overlap":_parse_int_setting("CHUNK_OVERLAP",config.get("chunk_overlap",os.environ.get("CHUNK_OVERLAP",DEFAULT_CHUNK_OVERLAP))),
    }
    if resolved["top_k"] <= 0:             raise ValueError("TOP_K must be > 0")
    if resolved["chunk_size"] <= 0:        raise ValueError("CHUNK_SIZE must be > 0")
    if resolved["chunk_overlap"] < 0:      raise ValueError("CHUNK_OVERLAP must be >= 0")
    if resolved["chunk_overlap"] >= resolved["chunk_size"]: raise ValueError("CHUNK_OVERLAP must be < CHUNK_SIZE")
    return resolved

Cargar Documentos

In [4]:
def load_documents(data_dir=DEFAULT_DATA_DIR):
    docs = []
    for subfolder, doc_type in DOC_TYPE_MAP.items():
        folder = os.path.join(data_dir, subfolder)
        if not os.path.isdir(folder):
            continue
        for path in globmod.glob(os.path.join(folder, "*.txt")):
            with open(path, "r", encoding="utf-8") as f:
                content = f.read()
            docs.append(Document(
                page_content=content,
                metadata={"source": path, "doc_type": doc_type},
            ))
    return docs

Split Documents

In [5]:
def split_documents(docs, chunk_size=DEFAULT_CHUNK_SIZE, chunk_overlap=DEFAULT_CHUNK_OVERLAP):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, add_start_index=True
    )
    return splitter.split_documents(docs)

Construir Index

In [6]:
def build_index(chunks, embedding_model):
    texts = [c.page_content for c in chunks]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index

Retrieval

In [7]:
def retrieve(query, index, model, chunks, k=DEFAULT_TOP_K):
    q = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q)
    scores, indices = index.search(q, k)
    return [
        {"text": chunks[i].page_content, "score": float(s), "metadata": chunks[i].metadata}
        for s, i in zip(scores[0], indices[0]) if i != -1
    ]


System Prompt

In [8]:
SYSTEM_PROMPT = """You are a helpful personal assistant with access to the user's \
private documents, including emails, notes, SMS messages, and calendar events.

When answering questions:
- Base your answers ONLY on the context retrieved from the user's documents.
- If the retrieved context does not contain enough information, say so clearly \
and do NOT make up information.
- Cite the document type and source when relevant.
- Keep answers concise and directly relevant to what was asked."""

Clase Asistente

In [9]:
class Assistant:
    def __init__(self, index, model, chunks, client, config=None):
        self.index, self.model, self.chunks, self.client = index, model, chunks, client
        self.config   = resolve_config(config)
        self.llm_model = self.config["model"]
        self.top_k     = self.config["top_k"]
        self.history: list[dict] = []

    def ask(self, question, k=None):
        results = retrieve(question, self.index, self.model, self.chunks, k=k or self.top_k)
        if results:
            ctx = "\n\n".join(
                f"[{i}] ({r['metadata']['doc_type']} – {r['metadata']['source']})\n{r['text']}"
                for i, r in enumerate(results, 1)
            )
            user_content = f"Relevant documents:\n\n{ctx}\n\n---\n\nUser question: {question}"
        else:
            user_content = f"No relevant documents found.\n\nUser question: {question}"
        messages = [{"role":"system","content":SYSTEM_PROMPT}] + self.history + [{"role":"user","content":user_content}]
        answer = self.client.chat.completions.create(model=self.llm_model, messages=messages).choices[0].message.content
        self.history += [{"role":"user","content":question}, {"role":"assistant","content":answer}]
        return answer

    def clear_history(self): self.history.clear()

    @classmethod
    def from_config(cls, config=None):
        cfg   = resolve_config(config)
        docs  = load_documents()
        chunks = split_documents(docs, cfg["chunk_size"], cfg["chunk_overlap"])
        emb_model = SentenceTransformer(cfg["embedding_model"])
        index = build_index(chunks, emb_model)
        kw = {}
        if cfg["api_key"]:  kw["api_key"]  = cfg["api_key"]
        if cfg["base_url"]: kw["base_url"] = cfg["base_url"]
        return cls(index, emb_model, chunks, OpenAI(**kw), cfg)

Cargar Datos

In [11]:
from google.colab import files
import os, zipfile

uploaded = files.upload()

with zipfile.ZipFile("data.zip", "r") as z:
    z.extractall(".")

for folder in ["emails", "notes", "sms", "calendar"]:
    path = f"data/{folder}"
    count = len(os.listdir(path)) if os.path.isdir(path) else 0
    print(f"  {path}: {count} files")

Saving data.zip to data.zip
  data/emails: 2 files
  data/notes: 2 files
  data/sms: 2 files
  data/calendar: 2 files


Cargar API key y especificar configs

In [12]:
from google.colab import userdata

config = {
    "api_key":         userdata.get("OPENAI_API_KEY"),
    "model":           "gpt-4.1-mini",
    "embedding_model": "all-MiniLM-L6-v2",
    "top_k":           4,
    "chunk_size":      256,
    "chunk_overlap":   32,
}

assistant = Assistant.from_config(config)
print("Assistant ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Assistant ready!


Chat loop


In [ ]:
import textwrap

print("""
╔══════════════════════════════════════════════════╗
║         Asistente Digital Personal               ║
║                                                  ║
║  Pregúntame sobre correos, notas, SMS y          ║
║  eventos del calendario.                         ║
║  Escribe '/clear' para reiniciar el historial.   ║
║  Escribe '/quit' para salir.                     ║
╚══════════════════════════════════════════════════╝
""")

while True:
    try:
        user_input = input("Tú: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n¡Hasta luego!"); break
    if not user_input: continue
    if user_input == "/quit": print("¡Hasta luego!"); break
    if user_input == "/clear": assistant.clear_history(); print("Historial borrado.\n"); continue
    answer = assistant.ask(user_input)
    wrapped = textwrap.fill(answer, width=80)
    print(f"\nAsistente: {wrapped}\n")


╔══════════════════════════════════════════════════╗
║         Asistente Digital Personal               ║
║                                                  ║
║  Pregúntame sobre correos, notas, SMS y          ║
║  eventos del calendario.                         ║
║  Escribe '/clear' para reiniciar el historial.   ║
║  Escribe '/quit' para salir.                     ║
╚══════════════════════════════════════════════════╝

Tú: Cuando es mi cita con el doctor?

Asistente: Tu cita con el doctor es el lunes 25 de mayo de 2026 a las 9:00 AM en el
consultorio ubicado en Blvd. Agua Caliente 4500. (Fuente: email y calendario)

Tú: De que hable con Rodrigo?

Asistente: Hablaste con Rodrigo sobre arreglar el carro, mencionando que era la batería y
que te costó 1,800 pesos en el taller de don Pepe en la Constitución. También
hablaron sobre ir el domingo a casa de tu mamá, donde habrá pozole, y Rodrigo
confirmó que asistirá. (Fuente: SMS con Rodrigo)

Tú: Que tengo de pendientes esta semana?

Asis